# CSV Generation Script for the Andrew+ Pipetting Robot — RNA Amount Normalization

Converts a nanodrop9000 export into a CSV you can paste directly into the Andrew+ "Normalize" pipetting method.

**Input**

- The nanodrop9000 export, using its `Well`, `Sample ID`, and `Conc.` columns.
- User-supplied values (set in Config, below):
    - **Unit** — the concentration unit. OneLab only accepts molarity, but this can be `ng/uL` if that's what you're working with.
    - **Description** — sample type (e.g. `RNA`, `DNA`).
    - **Sample** — sample flag so the Andrew+ recognizes the row as a sample.
    - **Volume** — starting volume (uL) of the nanodropped solution loaded onto the Andrew+.

**Output**

- A CSV file you can open and copy/paste directly into the OneLab software.

In [7]:
import pandas as pd
from pathlib import Path

## Paths

Sets up the file locations this notebook reads from and writes to.

- `HOME` — anchors everything to the `Automation` folder (your VS Code workspace root), so paths stay portable across machines.
- `nanodrop_file` (below, in Config) — the raw nanodrop9000 export to convert.
- `output_path` (below, in Run) — where the generated sample sheet CSV is saved.

For both of those: in the VS Code file explorer, right-click the file &rarr; **Copy Relative Path**, then paste the result directly into the quotes after `HOME /`.

In [ ]:
# Home directory - all other paths in this notebook are relative to this
HOME = Path("/Users/mszucs/Library/CloudStorage/OneDrive-NewEnglandBiolabs 2/New_England_Biolabs/Projects/2026_Auto_Automation_Development")

## Config

Edit these values per run:

- `nanodrop_file` — path to the nanodrop9000 export for this run (paste via **Copy Relative Path**, see Paths above).
- `Unit` — concentration unit. OneLab only accepts molarity, but this can be set to `ng/ul` if that's what you're working with.
- `Description` — sample type (e.g. `RNA`, `DNA`).
- `Sample` — sample flag so the Andrew+ recognizes the row as a sample.
- `Volume` — starting volume (uL) of the nanodropped solution that goes on the Andrew+.

In [ ]:
# Configuration for nanodrop file and sample sheet columns

nanodrop_file = HOME / "260914_Andrew_Plus_NucleosideDigestion/241021_xrRNA_VAI-1_TYMV_HPAMPD_DMS.txt"
Unit = "M"
Description = "RNA"
Sample = "Sample"
Volume = "30"

## Function

Defines `generate_sample_sheet`, which does the actual conversion:

1. Reads the nanodrop9000 export (tab-separated) into a DataFrame.
2. Pulls the `Well`, `Sample ID`, and `Conc.` columns and maps them to the Andrew+ "Normalize" method's expected columns (`Position`, `Solution name`, `Concentration`).
3. Fills in `Unit`, `Volume (uL)`, `Description`, and `Sample` from the Config values, applied to every row.
4. Writes the result to `output_path` as a CSV and returns it as a DataFrame for the preview below.

No values need editing here — this cell just defines the function; it runs when called in Run below.

In [10]:
def generate_sample_sheet(nanodrop_file, output_path, unit, volume, description, sample):
    nanodrop = pd.read_csv(nanodrop_file, sep="\t")

    output_columns = ["Position", "Solution name", "Concentration", "Unit", "Volume (uL)", "Description", "Sample"]
    sample_sheet = pd.DataFrame(columns=output_columns)
    sample_sheet["Position"] = nanodrop["Well "]
    sample_sheet["Solution name"] = nanodrop["Sample ID"]
    sample_sheet["Concentration"] = nanodrop["Conc. "]
    sample_sheet["Unit"] = unit
    sample_sheet["Volume (uL)"] = volume
    sample_sheet["Description"] = description
    sample_sheet["Sample"] = sample

    sample_sheet.to_csv(output_path, index=False)
    return sample_sheet

## Run

Sets `output_path` (paste via **Copy Relative Path**, see Paths above), calls `generate_sample_sheet` with the Config values, saves the CSV there, and previews the first few rows below.

Open the saved CSV and copy/paste its contents directly into the OneLab "Normalize" pipetting method.

In [ ]:
output_path = HOME / "260914_Andrew_Plus_NucleosideDigestion/Sample_Sheet.csv"

sample_sheet = generate_sample_sheet(
    nanodrop_file=nanodrop_file,
    output_path=output_path,
    unit=Unit,
    volume=Volume,
    description=Description,
    sample=Sample,
)
sample_sheet.head()